# 06 - Persistência do modelo: salvar, carregar e inferir

Este notebook roda em um **kernel limpo**: não importa nada do treinamento (`src.train`, `src.experiments`) e
não reutiliza variáveis dos notebooks anteriores. Ele usa apenas os arquivos `.pt` gerados no notebook 05 e o
módulo de inferência independente `src/inference.py`.

Conteúdo:
1. as duas formas de persistência (`state_dict` × modelo inteiro) e qual foi adotada;
2. conteúdo do checkpoint;
3. carregamento em ambiente limpo e inferência (no notebook e pela linha de comando);
4. verificação de fidelidade (saídas e métrica no teste idênticas às do modelo em memória);
5. `eval()` × `train()` com dropout; `map_location`; tamanho dos arquivos;
6. desafio opcional: TorchScript e quantização dinâmica.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import torch

from src import inference                # só o módulo de inferência
from src.models import MLP, n_parametros # a classe da arquitetura (necessária para recriar a rede)

MODELS = RAIZ / "models"
TABLES = RAIZ / "reports" / "tables"
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())
print("Variáveis do treinamento neste kernel?", any(n in globals() for n in ("dados", "modelo", "BASELINE")))

PyTorch 2.14.0+cpu | threads: 6
Variáveis do treinamento neste kernel? False


## 1. Duas formas de persistência

| | `torch.save(modelo.state_dict(), caminho)` | `torch.save(modelo, caminho)` |
|---|---|---|
| O que grava | só os tensores de pesos e vieses (dicionário nome → tensor) | o objeto Python inteiro, via `pickle` |
| Para carregar | recriar a arquitetura e chamar `load_state_dict` | `torch.load` reconstrói o objeto |
| Dependências | nenhuma do código; basta saber a arquitetura | a classe `MLP` precisa existir **no mesmo caminho de módulo** (`src.models.MLP`) |
| Robustez | estável entre versões e refatorações | quebra se a classe mudar de nome/pasta ou entre versões |
| Segurança | pode ser lido com `weights_only=True` | exige `weights_only=False`: o unpickler pode executar código arbitrário embutido no arquivo |

**Adotado:** `state_dict` dentro de um **checkpoint completo** (dicionário com pesos + configuração +
pré-processamento). Como o checkpoint só contém tensores, números, strings, listas e dicionários, ele é lido
com `torch.load(..., weights_only=True)`, que usa um unpickler restrito e recusa qualquer outro objeto. Isso
protege contra arquivos `.pt` maliciosos e é o padrão do `torch.load` desde o PyTorch 2.6.

Demonstração com o modelo de classificação: salvar o modelo inteiro e tentar abri-lo com `weights_only=True`.

In [2]:
modelo_clf, ckpt_clf = inference.carregar(MODELS / "modelo_classificacao.pt")
caminho_inteiro = MODELS / "_demo_modelo_inteiro.pt"
torch.save(modelo_clf, caminho_inteiro)
try:
    torch.load(caminho_inteiro, weights_only=True)
except Exception as e:
    print("weights_only=True recusou o modelo inteiro:", type(e).__name__)
    print(str(e).splitlines()[0][:200])
inteiro = torch.load(caminho_inteiro, weights_only=False)  # só para arquivo de origem confiável
print("Com weights_only=False carrega como", type(inteiro).__module__ + "." + type(inteiro).__name__)
tam_inteiro = os.path.getsize(caminho_inteiro) / 1024
torch.save(modelo_clf.state_dict(), MODELS / "_demo_state_dict.pt")
tam_sd = os.path.getsize(MODELS / "_demo_state_dict.pt") / 1024
print(f"Tamanho: modelo inteiro = {tam_inteiro:.1f} KB | só state_dict = {tam_sd:.1f} KB")
os.remove(caminho_inteiro); os.remove(MODELS / "_demo_state_dict.pt")

weights_only=True recusou o modelo inteiro: UnpicklingError
Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
Com weights_only=False carrega como src.models.MLP
Tamanho: modelo inteiro = 4.9 KB | só state_dict = 3.5 KB


O modelo inteiro carrega como `src.models.MLP`: o arquivo guarda o caminho do módulo, e não o código da classe.
Se a pasta `src` for renomeada, esse arquivo deixa de abrir; o `state_dict` continua válido.

## 2. Conteúdo do checkpoint

In [3]:
def descrever(ckpt):
    linhas = []
    for k, v in ckpt.items():
        if k == "state_dict":
            desc = ", ".join(f"{n} {tuple(t.shape)}" for n, t in v.items())
        elif isinstance(v, torch.Tensor):
            desc = f"tensor {tuple(v.shape)}: {np.round(v.numpy()[:3], 3).tolist()}..."
        elif isinstance(v, list) and len(v) > 4:
            desc = f"lista com {len(v)} itens: {v[:3]}..."
        elif isinstance(v, dict) and k == "historico":
            desc = f"perdas por época (treino/val), {len(v['treino'])} épocas"
        else:
            desc = str(v)
        linhas.append((k, desc[:160]))
    return pd.DataFrame(linhas, columns=["chave", "conteúdo"])

modelo_reg, ckpt_reg = inference.carregar(MODELS / "modelo_regressao.pt")
pd.set_option("display.max_colwidth", 170)
descrever(ckpt_reg)

,chave,conteúdo
0,state_dict,"net.0.weight (32, 30), net.0.bias (32,), net.3.weight (32, 32), net.3.bias (32,), net.6.weight (1, 32), net.6.bias (1,)"
1,config,"{'in_dim': 30, 'out_dim': 1, 'hidden': [32, 32], 'activation': 'Sigmoid', 'dropout': 0.1}"
2,hiperparametros,"{'neuronios': 32, 'camadas': 2, 'ativacao': 'Sigmoid', 'otimizador': 'AdamW', 'lr': 0.00016, 'batch': 16, 'epocas': 200, 'paciencia': 20, 'weight_decay': 0.0001"
3,problema,regressao
4,target,max_heart_rate_achieved
5,features,"lista com 30 itens: ['age', 'sex', 'resting_bp_systolic']..."
6,x_media,"tensor (30,): [54.039, 0.523, 127.887]..."
7,x_desvio,"tensor (30,): [12.945, 0.499, 13.736]..."
8,versao_torch,2.14.0+cpu
9,y_media,164.83072916666666


In [4]:
descrever(ckpt_clf)

,chave,conteúdo
0,state_dict,"net.0.weight (8, 30), net.0.bias (8,), net.3.weight (1, 8), net.3.bias (1,)"
1,config,"{'in_dim': 30, 'out_dim': 1, 'hidden': [8], 'activation': 'Tanh', 'dropout': 0.1}"
2,hiperparametros,"{'neuronios': 8, 'camadas': 1, 'ativacao': 'Tanh', 'otimizador': 'RMSprop', 'lr': 0.00162, 'batch': 16, 'epocas': 200, 'paciencia': 20, 'weight_decay': 0.0001,"
3,problema,classificacao
4,target,has_heart_disease
5,features,"lista com 30 itens: ['age', 'sex', 'resting_bp_systolic']..."
6,x_media,"tensor (30,): [54.039, 0.523, 127.887]..."
7,x_desvio,"tensor (30,): [12.945, 0.499, 13.736]..."
8,versao_torch,2.14.0+cpu
9,classes,"['sem_doenca', 'com_doenca']"


Sem `x_media`/`x_desvio` (e, na regressão, `y_media`/`y_desvio`), o modelo carregado receberia features na escala
errada e devolveria a FC máxima em unidades padronizadas. Sem `features`, uma mudança na ordem das colunas
trocaria silenciosamente o significado de cada entrada. Sem `limiar`, a classificação voltaria ao 0,5 em vez do
limiar escolhido na validação.

## 3. Carregamento em ambiente limpo e inferência

`inference.carregar` lê o arquivo com `map_location="cpu"` e `weights_only=True`, recria a `MLP` com
`ckpt["config"]`, chama `load_state_dict` e `eval()`. `inference.prever` aplica a padronização do checkpoint e
roda a rede dentro de `torch.inference_mode()`.

Novas amostras: `exemplos/novas_amostras.csv` (5 pacientes do conjunto de teste, com as colunas originais do dataset).

In [5]:
novas = pd.read_csv(RAIZ / "exemplos" / "novas_amostras.csv")
print(novas.shape)
pd.concat([novas[["patient_id", "age", "max_heart_rate_achieved", "has_heart_disease"]],
           inference.prever(modelo_reg, ckpt_reg, novas).round(1),
           inference.prever(modelo_clf, ckpt_clf, novas).round(3)], axis=1)

(5, 32)


,patient_id,age,max_heart_rate_achieved,has_heart_disease,max_heart_rate_achieved_previsto,prob_com_doenca,has_heart_disease_previsto,classe
0,7798,27,197,0,195.600006,0.009,0,sem_doenca
1,10,51,151,0,171.500000,0.305,0,sem_doenca
2,2820,67,161,0,156.000000,0.010,0,sem_doenca
3,8937,67,157,0,157.199997,0.154,0,sem_doenca
4,6217,65,144,1,140.600006,0.436,0,sem_doenca


Também é possível prever a partir de um dicionário (uma amostra):

In [6]:
amostra = novas.drop(columns=["patient_id", "max_heart_rate_achieved", "has_heart_disease"]).iloc[0].to_dict()
amostra["max_heart_rate_achieved"] = float(novas.loc[0, "max_heart_rate_achieved"])  # feature da classificação
amostra["has_heart_disease"] = int(novas.loc[0, "has_heart_disease"])              # feature da regressão
print(inference.prever(modelo_reg, ckpt_reg, amostra).round(2).to_dict("records"))
print(inference.prever(modelo_clf, ckpt_clf, amostra).round(4).to_dict("records"))

[{'max_heart_rate_achieved_previsto': 195.5800018310547}]
[{'prob_com_doenca': 0.008899999782443047, 'has_heart_disease_previsto': 0, 'classe': 'sem_doenca'}]


### Pela linha de comando (processo Python novo)

O script roda num processo separado, sem nada deste notebook em memória.

In [7]:
for arq in ("modelo_regressao.pt", "modelo_classificacao.pt"):
    cmd = [sys.executable, "src/inference.py", "--modelo", f"models/{arq}",
           "--entrada", "exemplos/novas_amostras.csv", "--saida", f"exemplos/previsoes_{arq[7:-3]}.csv"]
    print(">", " ".join(cmd[1:]))
    saida = subprocess.run(cmd, cwd=RAIZ, capture_output=True, text=True, encoding="utf-8")
    print(saida.stdout or saida.stderr)

> src/inference.py --modelo models/modelo_regressao.pt --entrada exemplos/novas_amostras.csv --saida exemplos/previsoes_regressao.csv


Modelo: models/modelo_regressao.pt (regressao, target = max_heart_rate_achieved)
 patient_id  max_heart_rate_achieved_previsto
       7798                        195.580536
         10                        171.458282
       2820                        156.049988
       8937                        157.183914
       6217                        140.608521
Previsões gravadas em exemplos/previsoes_regressao.csv

> src/inference.py --modelo models/modelo_classificacao.pt --entrada exemplos/novas_amostras.csv --saida exemplos/previsoes_classificacao.csv


Modelo: models/modelo_classificacao.pt (classificacao, target = has_heart_disease)
 patient_id  prob_com_doenca  has_heart_disease_previsto     classe
       7798         0.008899                           0 sem_doenca
         10         0.305097                           0 sem_doenca
       2820         0.010178                           0 sem_doenca
       8937         0.153696                           0 sem_doenca
       6217         0.436283                           0 sem_doenca
Previsões gravadas em exemplos/previsoes_classificacao.csv



## 4. Verificação de fidelidade

No notebook 05, logo antes de salvar, foram gravadas as saídas brutas do modelo **em memória** (logits na
classificação, valores padronizados na regressão) para a validação e o teste, e a métrica principal no teste.
Aqui o modelo **carregado do arquivo** é aplicado aos mesmos dados (refeitos a partir do CSV original com o
pré-processamento do checkpoint) e as saídas são comparadas com `torch.allclose`.

In [8]:
from sklearn.metrics import f1_score, mean_squared_error

linhas = []
for problema, (modelo, ckpt) in {"regressao": (modelo_reg, ckpt_reg),
                                 "classificacao": (modelo_clf, ckpt_clf)}.items():
    ref = torch.load(RAIZ / "logs" / f"saidas_em_memoria_{problema}.pt", weights_only=True)
    df = pd.read_csv(RAIZ / "data" / "processed" / f"dataset_{'regression' if problema == 'regressao' else 'classification'}.csv")
    for conjunto in ("val", "teste"):
        sub = df.set_index("patient_id").loc[ref[f"ids_{conjunto}"].tolist()].reset_index()
        x = inference.preparar_entrada(ckpt, sub)
        with torch.inference_mode():
            saida = modelo(x).squeeze(1)
        dif = (saida - ref[f"saida_{conjunto}"]).abs().max().item()
        linha = {"problema": problema, "conjunto": conjunto, "n": len(sub),
                 "allclose (atol=1e-6)": torch.allclose(saida, ref[f"saida_{conjunto}"], atol=1e-6),
                 "idênticas (equal)": torch.equal(saida, ref[f"saida_{conjunto}"]),
                 "dif. máx. absoluta": dif}
        if conjunto == "teste":
            y = sub[ckpt["target"]].to_numpy(dtype=float)  # float, como no treino
            if problema == "regressao":
                prev = inference.prever(modelo, ckpt, sub).iloc[:, 0].to_numpy()
                linha["métrica"] = "RMSE"
                linha["antes de salvar"] = ref["metrica_teste"]
                linha["modelo carregado"] = float(np.sqrt(mean_squared_error(y, prev)))
            else:
                prev = inference.prever(modelo, ckpt, sub)[f"{ckpt['target']}_previsto"].to_numpy()
                linha["métrica"] = "F1"
                linha["antes de salvar"] = ref["metrica_teste"]
                linha["modelo carregado"] = float(f1_score(y, prev))
        linhas.append(linha)
fidelidade = pd.DataFrame(linhas)
fidelidade.to_csv(TABLES / "fidelidade.csv", index=False)
fidelidade

,problema,conjunto,n,allclose (atol=1e-6),idênticas (equal),dif. máx. absoluta,métrica,antes de salvar,modelo carregado
0,regressao,val,1440,True,True,0.0,NaN,NaN,NaN
1,regressao,teste,1800,True,True,0.0,RMSE,12.164090,12.164090
2,classificacao,val,1440,True,True,0.0,NaN,NaN,NaN
3,classificacao,teste,1800,True,True,0.0,F1,0.821561,0.821561


As saídas do modelo carregado coincidem com as do modelo em memória (diferença máxima absoluta na tabela) e a
métrica no teste recalculada com o modelo carregado é igual à obtida antes de salvar. Os cuidados que garantem
isso: (1) recriar exatamente a mesma arquitetura (`config` no checkpoint); (2) aplicar a mesma padronização com as
estatísticas do treino guardadas no checkpoint, na mesma ordem de features; (3) `eval()` antes de prever;
(4) na classificação, usar o mesmo limiar.

## 5. O que acontece se esquecer o `eval()`

O modelo final não usa dropout (ver notebook 05), então `train()` e `eval()` dão o mesmo resultado para ele.
Para mostrar o efeito, treina-se rapidamente uma rede **com dropout 0,3** (mesma arquitetura do baseline, 30
épocas em lote único sobre a validação, só para a demonstração) e ela é aplicada 3 vezes aos mesmos 5 pacientes em cada modo.

In [9]:
torch.manual_seed(0)
# dados da demonstração: os 1.440 pacientes de validação (o teste não é usado)
df_clf = pd.read_csv(RAIZ / "data" / "processed" / "dataset_classification.csv").set_index("patient_id")
ref_clf = torch.load(RAIZ / "logs" / "saidas_em_memoria_classificacao.pt", weights_only=True)
val = df_clf.loc[ref_clf["ids_val"].tolist()]
xt = inference.preparar_entrada(ckpt_clf, val)
yt = torch.tensor(val["has_heart_disease"].to_numpy(), dtype=torch.float32).unsqueeze(1)
x_val = inference.preparar_entrada(ckpt_clf, pd.read_csv(RAIZ / "exemplos" / "novas_amostras.csv"))
rede_dropout = MLP(len(ckpt_clf["features"]), 1, (32, 32), "ReLU", dropout=0.3)
opt = torch.optim.Adam(rede_dropout.parameters(), lr=1e-3)
for _ in range(30):
    rede_dropout.train(); opt.zero_grad()
    torch.nn.functional.binary_cross_entropy_with_logits(rede_dropout(xt), yt).backward(); opt.step()

with torch.no_grad():
    rede_dropout.train()
    modo_treino = [torch.sigmoid(rede_dropout(x_val)).squeeze(1).numpy() for _ in range(3)]
    rede_dropout.eval()
    modo_eval = [torch.sigmoid(rede_dropout(x_val)).squeeze(1).numpy() for _ in range(3)]
pd.DataFrame({**{f"train() #{i+1}": p for i, p in enumerate(modo_treino)},
              **{f"eval() #{i+1}": p for i, p in enumerate(modo_eval)}}).round(3)

,train() #1,train() #2,train() #3,eval() #1,eval() #2,eval() #3
0,0.228,0.288,0.311,0.307,0.307,0.307
1,0.395,0.461,0.433,0.431,0.431,0.431
2,0.302,0.261,0.419,0.289,0.289,0.289
3,0.421,0.458,0.417,0.445,0.445,0.445
4,0.417,0.421,0.385,0.409,0.409,0.409


Em modo `train()`, o dropout continua sorteando neurônios desligados a cada chamada (e multiplica os restantes
por 1/(1-p)): o mesmo paciente recebe probabilidades diferentes a cada execução, e a previsão deixa de ser
reprodutível. Em `eval()`, o dropout vira identidade e a saída é determinística. Numa rede com batch
normalization, esquecer o `eval()` faria a camada normalizar pelas estatísticas do próprio lote de inferência,
e não pelas médias acumuladas no treino; com uma amostra por vez, o resultado fica sem sentido.
`torch.inference_mode()` (ou `no_grad()`) é complementar: não muda o comportamento das camadas, apenas deixa de
construir o grafo do autograd, o que economiza memória e tempo.

## 6. Portabilidade: `map_location`

Todo o treino foi em CPU (`torch.cuda.is_available()` = False nesta máquina). Se o modelo tivesse sido treinado em
GPU, os tensores do `state_dict` estariam salvos como `cuda:0`, e `torch.load` sem `map_location` tentaria
recriá-los na GPU, falhando numa máquina sem CUDA. Com `map_location="cpu"` eles são remapeados para a CPU ao
carregar. O checkpoint salvo a partir da CPU também pode ser levado a uma GPU com `map_location="cuda"` ou `modelo.to("cuda")`.

In [10]:
print("CUDA disponível:", torch.cuda.is_available())
ck = torch.load(MODELS / "modelo_regressao.pt", map_location="cpu", weights_only=True)
print("Dispositivo dos pesos carregados:", {t.device for t in ck["state_dict"].values()})
# simulação: um checkpoint cujos tensores apontam para outro dispositivo é remapeado na leitura
ck2 = torch.load(MODELS / "modelo_regressao.pt", map_location=lambda storage, loc: storage, weights_only=True)
print("map_location por função (mantém onde estava salvo):", {t.device for t in ck2["state_dict"].values()})

CUDA disponível: False
Dispositivo dos pesos carregados: {device(type='cpu')}
map_location por função (mantém onde estava salvo): {device(type='cpu')}


## 7. Tamanho dos arquivos

Cada parâmetro em float32 ocupa 4 bytes. A diferença entre o tamanho do arquivo e parâmetros × 4 é o overhead:
a estrutura do zip do `torch.save`, nomes das chaves, pickle do dicionário, estatísticas de padronização,
lista de features, histórico de perdas e hiperparâmetros.

In [11]:
linhas = []
for arq in ["baseline_regressao.pt", "modelo_regressao.pt", "baseline_classificacao.pt", "modelo_classificacao.pt"]:
    modelo, ckpt = inference.carregar(MODELS / arq)
    n = n_parametros(modelo)
    tam = os.path.getsize(MODELS / arq) / 1024
    torch.save(modelo.state_dict(), MODELS / "_tmp.pt")
    tam_sd = os.path.getsize(MODELS / "_tmp.pt") / 1024
    os.remove(MODELS / "_tmp.pt")
    linhas.append({"arquivo": arq, "camadas ocultas": ckpt["config"]["hidden"], "parâmetros": n,
                   "parâmetros × 4 B (KB)": n * 4 / 1024, "só state_dict (KB)": tam_sd,
                   "checkpoint completo (KB)": tam,
                   "overhead (KB)": tam - n * 4 / 1024})
tamanhos = pd.DataFrame(linhas)
tamanhos.to_csv(TABLES / "tamanho_arquivos.csv", index=False)
tamanhos.round(2)

,arquivo,camadas ocultas,parâmetros,parâmetros × 4 B (KB),só state_dict (KB),checkpoint completo (KB),overhead (KB)
0,baseline_regressao.pt,"[32, 32]",2081,8.13,10.80,13.91,5.78
1,modelo_regressao.pt,"[32, 32]",2081,8.13,10.80,16.14,8.01
2,baseline_classificacao.pt,"[32, 32]",2081,8.13,10.80,14.09,5.96
3,modelo_classificacao.pt,[8],257,1.00,3.31,6.36,5.36


## 8. Desafio opcional: TorchScript e quantização dinâmica

Aplicado ao modelo de classificação:
- **TorchScript** (`torch.jit.trace`): grava o grafo da rede, que pode ser carregado sem a classe Python
  (`torch.jit.load`), inclusive em C++.
- **Quantização dinâmica** (`quantize_dynamic`, int8): os pesos das camadas `Linear` passam a 8 bits e as
  ativações são quantizadas em tempo de execução.

Comparação de tamanho, latência (conjunto de teste inteiro, 30 repetições com aquecimento) e F1 no teste.

In [12]:
import time
import warnings

warnings.filterwarnings("ignore")
ref = torch.load(RAIZ / "logs" / "saidas_em_memoria_classificacao.pt", weights_only=True)
df_clf = pd.read_csv(RAIZ / "data" / "processed" / "dataset_classification.csv")
teste = df_clf.set_index("patient_id").loc[ref["ids_teste"].tolist()].reset_index()
x_te = inference.preparar_entrada(ckpt_clf, teste)
y_te = teste["has_heart_disease"].to_numpy()

def latencia(f, x, n_rep=30, n_aq=5):
    with torch.inference_mode():
        for _ in range(n_aq):
            f(x)
        t = []
        for _ in range(n_rep):
            i = time.perf_counter(); f(x); t.append(time.perf_counter() - i)
    return 1000 * np.mean(t), 1000 * np.std(t)

variantes = {}
variantes["original (float32)"] = modelo_clf
script = torch.jit.trace(modelo_clf, x_te[:10])
torch.jit.save(script, MODELS / "_ts.pt")
variantes["TorchScript"] = torch.jit.load(MODELS / "_ts.pt")
quant = torch.ao.quantization.quantize_dynamic(modelo_clf, {torch.nn.Linear}, dtype=torch.qint8)
torch.save(quant.state_dict(), MODELS / "_q.pt")
variantes["quantizado int8"] = quant

linhas = []
for nome, m in variantes.items():
    with torch.inference_mode():
        prob = torch.sigmoid(m(x_te)).squeeze(1).numpy()
    lat, lat_sd = latencia(m, x_te)
    arq = {"original (float32)": MODELS / "modelo_classificacao.pt", "TorchScript": MODELS / "_ts.pt",
           "quantizado int8": MODELS / "_q.pt"}[nome]
    linhas.append({"variante": nome, "tamanho (KB)": os.path.getsize(arq) / 1024,
                   "latência teste inteiro (ms)": f"{lat:.3f} ± {lat_sd:.3f}",
                   "F1 teste": f1_score(y_te, (prob >= ckpt_clf["limiar"]).astype(int)),
                   "dif. máx. prob. vs original": float(np.abs(prob - torch.sigmoid(modelo_clf(x_te)).squeeze(1).detach().numpy()).max())})
for f in ("_ts.pt", "_q.pt"):
    os.remove(MODELS / f)
otimizacao = pd.DataFrame(linhas)
otimizacao.to_csv(TABLES / "torchscript_quantizacao.csv", index=False)
otimizacao.round(5)

,variante,tamanho (KB),latência teste inteiro (ms),F1 teste,dif. máx. prob. vs original
0,original (float32),6.36035,0.133 ± 0.035,0.82156,0.00000
1,TorchScript,11.76270,0.133 ± 0.006,0.82156,0.00000
2,quantizado int8,3.63184,0.331 ± 0.009,0.82004,0.05563
